[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forestdatapartnership/whisp/blob/main/notebooks/timber_pathway_viewer.ipynb)

# Whisp timber: interactive decision-tree / map viewer

This notebook builds an **interactive viewer** that links the Whisp **timber decision tree** (a Mermaid flowchart) to a **Leaflet map**, so you can explore how the timber risk pathway is built up from the underlying Earth Engine layers over Brazil.

**What it shows**
- Click a **2020 question** node to map that land class's 2020 source-agreement layer (how many input datasets agree the pixel is that class).
- Click a **2025 question** node to map the after-2020 gain / change layer.
- Click a **verdict terminal** to pick a feeding pathway code and map it in its palette colour.
- A **final-verdict** result map (green = low / amber = more-info / red = high) is shown by default, with solid/pale confidence shading.
- Toggles: **dilate** (zoom-stable bloom), **OR vs k>=2** agreement, Sentinel-2 2020 / 2024-25 before-after backgrounds, base map, and demo-site bookmarks.

Deforestation-risk frameworks such as the EU Deforestation Regulation (EUDR) are one example use; this is a general deforestation-risk view.

**How to run**
1. Open in Google Colab (badge above) or in a local Jupyter with the Whisp package installed.
2. Run the **Setup** cell to install `openforis-whisp`.
3. Set `PROJECT` to your own Earth Engine cloud project in the **Earth Engine init** cell, then run it (it will prompt you to authenticate the first time).
4. Run the **build** cell (resolves every map layer to a signed tile URL), then the **render** cell.

**Requirements:** a Google Earth Engine account and a registered cloud project. If unsure how to find your project id, see https://developers.google.com/earth-engine/cloud/assets

> **Note on tiles:** the Earth Engine tile URLs embedded in the map are *signed* and *expire* after some hours (Earth Engine does not publish an exact lifetime). This is inherent to `getMapId`. Just **re-run the build + render cells** to refresh them before a demo. The robustness here is about Earth Engine **initialisation**, not tile longevity.

## Setup: install and import packages

In [ ]:
# Install openforis-whisp (Colab). On a local environment where it is already installed this is a no-op.
# The '--pre' picks up pre-release builds, matching the other Whisp example notebooks.
try:
    import openforis_whisp  # noqa: F401
    print('openforis-whisp already importable')
except ImportError:
    import sys
    !{sys.executable} -m pip install --pre openforis-whisp

In [ ]:
import json
import os
import ee
from IPython.display import HTML, display

## Earth Engine initialisation (robust, configurable)

Set `PROJECT` below to **your own** Earth Engine cloud project id. There is no hidden hardcoded project and no `.env` file: you control which project is used.

The init tries `ee.Initialize(project=PROJECT)`; if that fails it runs `ee.Authenticate()` and retries, and gives a clear error if it still cannot connect. (Set `PROJECT` to your own Earth Engine Cloud project id before running.)

In [ ]:
# >>> SET THIS to your own Earth Engine cloud project id <<<
# Set PROJECT to your own Google Earth Engine Cloud project id, e.g. 'my-ee-project'.
PROJECT = os.environ.get('EE_PROJECT', '<your_gee_cloud_project>')

HIGH_VOLUME = 'https://earthengine-highvolume.googleapis.com'


def init_earthengine(project):
    """Robust, self-contained EE init: try Initialize, else Authenticate + retry, else explain."""
    if not project or project in ('your-ee-project-id', 'your_cloud_project_name'):
        raise ValueError(
            'Set PROJECT to your own Earth Engine cloud project id before running this cell.'
        )
    try:
        ee.Initialize(project=project, opt_url=HIGH_VOLUME)
    except Exception as first_err:  # noqa: BLE001
        print('First ee.Initialize failed (%s); authenticating...' % type(first_err).__name__)
        try:
            ee.Authenticate()
            ee.Initialize(project=project, opt_url=HIGH_VOLUME)
        except Exception as second_err:  # noqa: BLE001
            raise RuntimeError(
                "Earth Engine init failed for project '%s'.\n"
                'Check: (1) PROJECT is a cloud project YOU can access, '
                '(2) you completed ee.Authenticate() / are logged in, '
                '(3) the Earth Engine API is enabled for the project.\n'
                'Underlying error: %s' % (project, second_err)
            ) from second_err
    print('Earth Engine initialised on project:', project)


init_earthengine(PROJECT)

## Build the viewer layers

This cell embeds the viewer logic (adapted from the internal build script, which lives in a git-ignored folder so it cannot be imported here). It reads the dataset lookup via `openforis_whisp.risk` and the per-dataset Earth Engine images via `openforis_whisp.datasets` (both shipped in the installable package), reconstructs the timber pathway, and resolves every map layer to a **signed XYZ tile template** via `getMapId(...)['tile_fetcher'].url_format`.

All layers are clipped to Brazil (the demo extent). This cell makes many Earth Engine calls, so it can take a minute or two.

In [ ]:
# ============================================================================
# Whisp timber pathway viewer: layer build logic (embedded; self-contained).
# Adapted from the internal build script. Reads the lookup + datasets from the
# installed openforis-whisp package, rebuilds the timber pathway, and resolves
# each layer to a signed XYZ tile template (the tile token expires; re-run to refresh).
# ============================================================================
from openforis_whisp import datasets as d
from openforis_whisp.risk import lookup_gee_datasets_df as lut
from openforis_whisp.risk import (
    get_cols_ind_01_treecover,
    get_cols_ind_02_commodities,
    get_cols_ind_04_dist_after_2020,
    get_cols_ind_05_primary_2020,
    get_cols_ind_06_nat_reg_2020,
    get_cols_ind_07a_planted_2020,
    get_cols_ind_07b_plantation_2020,
    get_cols_ind_08b_plantation_after_2020,
    get_cols_ind_09_treecover_after_2020,
    get_cols_ind_10_agri_after_2020,
    get_cols_ind_12_other_land_2020,
    get_cols_ind_13_other_land_after_2020,
    get_cols_ind_14_primary_2025,
    get_cols_ind_15_agriculture_2020,
    get_cols_ind_16_plantation_presence_2025,
)

brazil = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Brazil"))


def s2(start, end):  # Sentinel-2 true-colour median (ground reference under the risk/agreement layers)
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(brazil).filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).median()
    )


_S2_VIS = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}


# --- band / union / count helpers --------------------------------------------
def band_img(r):  # select by INDEX, global 0-fill so a limited-footprint member does not mask the union
    fn = getattr(d, str(r["corresponding_variable"]), None)
    if fn is None:
        return None
    try:
        return ee.Image(fn()).select(0).gt(0).unmask(0, False)
    except Exception:  # noqa: BLE001
        return None


def union(mask):
    imgs = [band_img(r) for _, r in lut[mask].iterrows() if band_img(r) is not None]
    if not imgs:
        return ee.Image(0)
    u = imgs[0]
    for im in imgs[1:]:
        u = u.Or(im)
    return u


def count(rows):
    imgs = [band_img(r) for r in rows if band_img(r) is not None]
    c = ee.Image(0)
    for im in imgs:
        c = c.add(im)
    return c


def flag(names):  # boolean mask over lut for rows whose name is in `names` (a whisp getter's result list)
    return lut["name"].isin(list(names))


# Resolve each whisp per-indicator getter once against the lookup (the names whisp feeds per indicator).
_n_ind01 = get_cols_ind_01_treecover(lut)
_n_ind02 = get_cols_ind_02_commodities(lut, risk_col="use_for_risk_pcrop")
_n_ind04 = get_cols_ind_04_dist_after_2020(lut)
_n_ind05 = get_cols_ind_05_primary_2020(lut)
_n_ind06 = get_cols_ind_06_nat_reg_2020(lut)
_n_ind07a = get_cols_ind_07a_planted_2020(lut)
_n_ind07b = get_cols_ind_07b_plantation_2020(lut)
_n_ind08b = get_cols_ind_08b_plantation_after_2020(lut)
_n_ind09 = get_cols_ind_09_treecover_after_2020(lut)
_n_ind10 = get_cols_ind_10_agri_after_2020(lut)
_n_ind12 = get_cols_ind_12_other_land_2020(lut)
_n_ind13 = get_cols_ind_13_other_land_after_2020(lut)
_n_ind14 = get_cols_ind_14_primary_2025(lut)
_n_ind15 = get_cols_ind_15_agriculture_2020(lut)
_n_ind16 = get_cols_ind_16_plantation_presence_2025(lut)

# Flag-wired pools (each built from exactly whisp's flagged input rows for the mapped indicator).
primary_rows = [r for _, r in lut[flag(_n_ind05)].iterrows()]
ag_rows = [r for _, r in lut[flag(_n_ind10)].iterrows()]
primary_count = count(primary_rows)
ag_count = count(ag_rows)

a = union(flag(_n_ind02) | flag(_n_ind15))             # Rule-1 agriculture-2020 (commodities OR agriculture_2020)
r20 = union(flag(_n_ind06)).Or(union(flag(_n_ind07a)))  # regen = nat-reg + planted
pl20 = union(flag(_n_ind07b))                          # plantation 2020
r25 = union(flag(_n_ind09))                            # treecover after 2020
pl25 = union(flag(_n_ind08b))                          # plantation gain (degradation)
pl25_presence = union(flag(_n_ind16))                  # multi-year plantation presence 2021-2024
ol25 = union(flag(_n_ind13))                           # other land after 2020
ol20 = union(flag(_n_ind12))                           # other land 2020
tc20 = union(flag(_n_ind01))                           # treecover gate (mirrors risk.py)
_dist_timber_mask = (
    (lut["use_for_risk_timber"] == 1)
    & (lut["theme_timber"] == "disturbance_after")
    & (lut["exclude_from_output"] != 1)
)
dist = union(_dist_timber_mask)                        # timber-specific disturbance after 2020
ind14 = union(flag(_n_ind14))                          # primary_2025 candidate (inert in pathway)


# --- pathway() : the timber decision tree as a per-pixel pathway-code image ---
def pathway(p20, ag):
    p25 = p20.And(dist.Not())
    r20x = r20.And(p20.Not())
    regen_low_pre = r25.And(tc20).And(pl25.Not())
    regen_pl = r20x.And(regen_low_pre.Not()).And(pl25)               # regen -> plantation (code 7)
    regen_low = r20x.And(regen_low_pre)                              # regen stayed forest (code 3)
    regen_other = r20x.And(regen_low_pre.Not()).And(pl25.Not()).And(ol25)       # regen 2020 -> other land (code 17)
    regen_more = r20x.And(regen_low_pre.Not()).And(pl25.Not()).And(ol25.Not())  # regen, no 2025 (code 10)
    still_p = p25.And(pl25.Not())  # MISSED-HIGH fix (mirrors risk.py Rule 5): still primary EXCLUDES a new plantation
    prim_low = p20.And(still_p)                                     # still primary (code 2)
    prim_other = p20.And(still_p.Not()).And(ol25)                   # primary 2020 -> other land (code 16)
    prim_deg = p20.And(still_p.Not()).And(ol25.Not()).And(pl25)     # primary -> plantation (code 12)
    prim_more = p20.And(still_p.Not()).And(ol25.Not()).And(pl25.Not())  # primary, no 2025 (code 9)
    pl_stable = pl20.And(pl25_presence)                             # stable plantation (code 4)
    pl_other = pl20.And(pl25_presence.Not()).And(ol25)             # plantation 2020 -> other land (code 18)
    pl_more = pl20.And(pl25_presence.Or(ol25).Not())               # plantation 2020, no 2025 (code 8)
    path = ee.Image(5)  # default: MORE INFO
    path = path.where(prim_low, 2)
    path = path.where(prim_deg, 12)
    path = path.where(prim_more, 9)
    path = path.where(regen_pl, 7)
    path = path.where(regen_low, 3)
    path = path.where(regen_more, 10)
    path = path.where(pl_stable, 4)
    path = path.where(pl_more, 8)
    path = path.where(prim_other, 16)   # primary 2020 -> other land (was shared 14)
    path = path.where(regen_other, 17)  # regen 2020 -> other land (was shared 14)
    path = path.where(pl_other, 18)     # plantation 2020 -> other land (was shared 14)
    path = path.where(ag.And(tc20), 6)
    path = path.where(a, 1)
    path = path.where(ol20.And(ol25), 11)
    path = path.where(ol20.And(ol25.Not()), 13)
    return path


PALETTE = ["c7e9c0", "238b45", "41ab5d", "addd8e", "bcbddc", "e31a1c",
           "e7298a", "9e9ac8", "6a51a3", "807dba", "66c2a4", "67000d", "54278f", "238b8b", "8c8c9e",
           "80cdc1", "35978f", "01665e"]
_PALETTE_BY_CODE = {i + 1: PALETTE[i] for i in range(len(PALETTE))}

_CODE_NAMES = {
    1: "agriculture-2020 (LOW)", 2: "still-primary (LOW)",
    3: "regen stayed forest (LOW)",
    4: "stable plantation (LOW)", 5: "unclassified / more-info", 6: "deforestation forest->ag (HIGH)",
    7: "degradation regen/planted->plantation (HIGH)", 8: "plantation-2020 no-2025 (more-info)",
    9: "primary no-2025 (more-info)", 10: "regen no-2025 (more-info)", 11: "other-land-2020 stable (LOW)",
    12: "degradation primary->plantation (HIGH)", 13: "other-land-2020 changed (more-info)",
    14: "forest/plantation 2020 -> other land (LOW)",
    15: "regen -> matured to primary (data gap; empty in verdict)",
    16: "low: primary 2020 -> other land",
    17: "low: regen 2020 -> other land",
    18: "low: plantation 2020 -> other land",
}

v_or = pathway(primary_count.gte(1), ag_count.gte(1))    # OR-union: any 1 product fires the node
v_conv = pathway(primary_count.gte(2), ag_count.gte(2))  # k>=2 agreement on primary + agriculture

# Verdict class-code sets: LOW / HIGH / more-info.
LOW_CODES = [1, 2, 3, 4, 11, 16, 17, 18]
HIGH_CODES = [6, 7, 12]
MORE_CODES = [5, 8, 9, 10, 13]
_VERDICT_PALETTE = {"low": "41ab5d", "more": "f08c00", "high": "e31a1c"}
_VERDICT_VIS = {"min": 1, "max": 3, "palette": [_VERDICT_PALETTE["low"], _VERDICT_PALETTE["more"], _VERDICT_PALETTE["high"]]}


def _codes_mask(v_img, codes):
    m = ee.Image(0)
    for _c in codes:
        m = m.Or(v_img.eq(_c))
    return m


def _verdict3(v_img):  # 14-code pathway image -> 1=LOW / 2=more-info / 3=HIGH
    low = _codes_mask(v_img, LOW_CODES)
    more = _codes_mask(v_img, MORE_CODES)
    high = _codes_mask(v_img, HIGH_CODES)
    out = ee.Image(0)
    out = out.where(low, 1)
    out = out.where(more, 2)
    out = out.where(high, 3)
    return out.selfMask()


verdict3_or = _verdict3(v_or)
verdict3_conv = _verdict3(v_conv)

# CONFIDENCE: solid where >=2 products agree AND no cross-class 2020 overlap; pale elsewhere.
p20o = primary_count.gte(1)
_class2020_sum = (a.gt(0).add(r20.gt(0)).add(pl20.gt(0)).add(p20o.gt(0)).add(ol20.gt(0)))
overlap2020 = _class2020_sum.gt(1)
_agreed = primary_count.gte(2).Or(ag_count.gte(2))
solid_mask = _agreed.And(overlap2020.Not())

# --- zoom-stable dilation (fixed-scale reproject) ----------------------------
DILATE_KM = 4.0
_FIXED_M = 400
_DILATE_K = max(1, round(DILATE_KM * 1000 / _FIXED_M))


def _dilate(mask):  # focal_max over a fixed-scale circular kernel, constant at every zoom
    return (
        mask.selfMask()
        .focalMax(radius=_DILATE_K, kernelType="circle", units="pixels")
        .reproject(crs="EPSG:4326", scale=_FIXED_M)
    )


# --- 2020 agreement counts (flag-wired) --------------------------------------
_regen_rows = [r for _, r in lut[flag(_n_ind06) | flag(_n_ind07a)].iterrows()]
_plantation_rows = [r for _, r in lut[flag(_n_ind07b)].iterrows()]
_otherland_rows = [r for _, r in lut[flag(_n_ind12)].iterrows()]
_regen_count = count(_regen_rows)
_plantation_count = count(_plantation_rows)
_otherland_count = count(_otherland_rows)

# Split the 2020 agriculture agreement into cropland + tree-crop (pasture taken from the dedicated block).
_CROPLAND_NAMES = [
    "Soy_Song_2020", "nBR_INPE_TCamz_cer_annual_2020", "nBR_MapBiomas_col10_soy_2020",
    "nBR_MapBiomas_col10_annual_crops_2020", "GLAD_cropland_2020",
]
_TREECROP_NAMES = [
    "TMF_plant", "Oil_palm_Descals", "Oil_palm_FDaP", "Coffee_FDaP", "Cocoa_FDaP", "Cocoa_ETH",
    "Rubber_FDaP", "Rubber_RBGE", "ForTy_tree_crops_2020", "nCO_ideam_eufo_commission_2020",
    "nBR_INPE_TCamz_cer_perennial_2020", "nBR_MapBiomas_col10_coffee_2020",
    "nBR_MapBiomas_col10_palmoil_2020", "nBR_MapBiomas_col10_pc_2020", "nCI_Cocoa_bnetd",
]
_cropland_rows = [r for _, r in lut[lut["name"].isin(_CROPLAND_NAMES)].iterrows()]
_treecrop_rows = [r for _, r in lut[lut["name"].isin(_TREECROP_NAMES)].iterrows()]
_cropland_count = count(_cropland_rows)
_treecrop_count = count(_treecrop_rows)

# Pasture 2020 agreement: MapBiomas class 15 + INPE TerraClass + Global Pasture Watch.
_MB10 = ee.Image(
    "projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_integration_v1")


def _m01p(img):
    return ee.Image(img).select(0).gt(0).unmask(0, False)


_p_mb_2020 = _m01p(_MB10.select("classification_2020").eq(15))
_p_tcamz = ee.Image("projects/ee-whisp/assets/NBR/terraclass_amz_2020")
_p_tccer = ee.Image("projects/ee-whisp/assets/NBR/terraclass_cer_2020")
_p_inpe_2020 = _m01p(_p_tcamz.eq(10).Or(_p_tcamz.eq(11))).Or(_m01p(_p_tccer.eq(11)))
_p_GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")
_p_gpw_dc = lambda yr: ee.Image(_p_GPW.filter(ee.Filter.eq("system:index", yr)).first()).select("dominant_class")
_p_gpw_2020 = _m01p(_p_gpw_dc("2020").eq(1))
_pasture_count = _p_mb_2020.add(_p_inpe_2020).add(_p_gpw_2020)
_p_mb_2024 = _m01p(_MB10.select("classification_2024").eq(15))
_p_gpw_2022 = _m01p(_p_gpw_dc("2022").eq(1))
_p_mb_gain = _p_mb_2024.And(_p_mb_2020.Not())
_p_gpw_gain = _p_gpw_2022.And(_p_gpw_2020.Not())
_pasture_gain = _p_mb_gain.Or(_p_gpw_gain)

# Agriculture as a whole 2020: each distinct ag source counts once, plus GPW (the one extra pasture source).
_ag2020_rows = [r for _, r in lut[flag(_n_ind02) | flag(_n_ind15)].iterrows()]
_ag2020_count = count(_ag2020_rows)
_ag_whole_count = _ag2020_count.add(_p_gpw_2020)
_AG_WHOLE_N = len(_ag2020_rows) + 1


def _row_img(name):
    return union(lut["name"] == name)


# After-2020 gain layers for the 2025 nodes.
_cropland_gain = _row_img("ESRI_crop_gain_2020_2025").Or(_row_img("GLAD_crop_gain_2020_2024"))
_treecrop_gain = _row_img("FDaP_tree_crop_gain_2020_2024")
_ag_whole_gain = union(flag(_n_ind10))                 # the deforestation `ag` union (Ind_10)
_plantation_gain = pl25
_other_land_gain = ol25.And(ol20.Not())
still_primary = p20o.And(dist.Not())
regen_stayed = r20.And(p20o.Not()).And(r25)

GAIN = {
    "cropland_gain": (_cropland_gain, "fe9929", "cropland gain after 2020 (annual/temporary; ESRI+GLAD)"),
    "treecrop_gain": (_treecrop_gain, "df65b0", "tree-crop gain after 2020 (perennial; FDaP palm/cocoa/rubber/coffee)"),
    "pasture_gain": (_pasture_gain, "78c679", "pasture gain after 2020 (MapBiomas 2020->2024 + GPW 2020->2022)"),
    "ag_whole_gain": (_ag_whole_gain, "d9880f", "agriculture gain after 2020 (all sources, the deforestation input)"),
    "plantation_gain": (_plantation_gain, "e7298a", "plantation expansion after 2020 (MapBiomas silviculture gain 2020->2024)"),
    "other_land_gain": (_other_land_gain, "ff7f00", "other-land gain after 2020 (new mining / built / water / rock / sand / salt)"),
    "plantation_presence": (pl25_presence, "41ab5d", "plantation presence 2021-2024 (still a plantation; MapBiomas silviculture any year)"),
    "forest_present_2025": (r25, "238b45", "forest present 2025 (treecover after 2020: TMF regrowth + ESRI 2025 treecover)"),
    "still_primary_2025": (still_primary, "08519c", "still primary 2025 (primary 2020 minus disturbance)"),
    "regen_stayed_2025": (regen_stayed, "7bccc4", "regen stayed forest 2025 (regen 2020 minus primary, still forest)"),
    "other_land_present_2025": (ol25, "7b6f5a", "other land present 2025 (other_land_after_2020 presence: ESRI 2025 + MapBiomas 2024 + mining after)"),
}

_RAMP2 = lambda c1, c2: {"min": 1, "max": 2, "palette": [c1, c2]}
_RAMP3 = lambda c1, c2, c3: {"min": 1, "max": 3, "palette": [c1, c2, c3]}

AGREEMENT = {
    "primary": (primary_count, _RAMP3("dbeecf", "74c476", "238b45"), len(primary_rows),
                "primary 2020 agreement (k>=1 / k>=2 / k>=3)"),
    "regen": (_regen_count.updateMask(p20o.Not()), _RAMP3("d7efd0", "7bccc4", "2b8cbe"), len(_regen_rows),
              "regen/planted 2020 agreement, EXCL primary (k>=1 / k>=2 / k>=3; matches the verdict carve-out)"),
    "plantation": (_plantation_count, _RAMP2("e5f5c9", "a1d99b"), len(_plantation_rows),
                   "plantation 2020 agreement (k>=1 / k>=2)"),
    "cropland": (_cropland_count, _RAMP3("fef0c8", "fdbb84", "d94801"), len(_cropland_rows),
                 "cropland 2020 agreement (annual/temporary; k>=1 / k>=2 / k>=3)"),
    "treecrop": (_treecrop_count, _RAMP3("f1e2cc", "c994c7", "980043"), len(_treecrop_rows),
                 "tree-crop 2020 agreement (perennial; k>=1 / k>=2 / k>=3)"),
    "pasture": (_pasture_count, _RAMP3("e5f0c0", "addd8e", "78c679"), 3,
                "pasture 2020 agreement (k>=1 / k>=2 / k>=3; MapBiomas+INPE+GPW)"),
    "ag_whole": (_ag_whole_count, {"min": 1, "max": _AG_WHOLE_N, "palette": ["f7e8c0", "e6b84d", "b8860b"]},
                 _AG_WHOLE_N, "agriculture-as-a-whole 2020 , count of distinct agriculture sources agreeing (cropland + tree-crop + pasture, incl. GPW)"),
    "other_land": (_otherland_count, _RAMP2("d8cdb8", "7b6f5a"), len(_otherland_rows),
                   "other land 2020 agreement (k>=1 / k>=2)"),
}


# --- resolve every layer to its signed XYZ tile template ---------------------
def _xyz(img, vis):  # getMapId -> the signed XYZ tile template (url_format). The token in this URL expires.
    return ee.Image(img).getMapId(vis)["tile_fetcher"].url_format


# --- PARALLELIZED tile-URL resolution ----------------------------------------
# getMapId is I/O-bound (one server round-trip per call) and EE's python client is thread-safe for it,
# so we resolve every (raw + dilated) tile template CONCURRENTLY via a thread pool instead of serially.
# We first DECLARE every job (a deferred ee image + vis under a unique slot id), resolve them all in
# parallel, then assemble the registry in the SAME order/keys as before from the resolved url map. This
# keeps the output config byte-for-byte equivalent to the serial build; only the resolution is parallel.
from concurrent.futures import ThreadPoolExecutor
import time as _time

_jobs = {}        # slot_id -> (ee_image, vis)  : every distinct tile template to resolve
_resolved = {}    # slot_id -> url             : filled in parallel below
_errors = {}      # slot_id -> exception       : surfaced after, never silently dropped


def _job(slot_id, img, vis):
    _jobs[slot_id] = (img, vis)
    return slot_id


# (1) Pathway layers for BOTH verdicts: code 1..14, raw + dilated, each in the code's palette colour.
_pathway_specs = []  # (key, hex, raw_slot, dil_slot, label)
for _verdict_img, _prefix, _tag in [(v_conv, "pathconv", "k>=2"), (v_or, "pathor", "OR")]:
    for _code in range(1, 19):
        if _code in (14, 15): continue  # skip retired 14 and the empty maturation slot 15
        _hex = _PALETTE_BY_CODE[_code]
        _mask = _verdict_img.eq(_code)
        _key = "%s_%d" % (_prefix, _code)
        _vis = {"palette": [_hex]}
        _rs = _job(_key + "::raw", _mask.selfMask().clip(brazil), _vis)
        _ds = _job(_key + "::dil", _dilate(_mask).clip(brazil), _vis)
        _pathway_specs.append((_key, _hex, _rs, _ds,
                               "pathway code %d (%s): %s" % (_code, _tag, _CODE_NAMES[_code])))

# (2) Agreement layers (graduated count), raw + dilated.
_agreement_specs = []  # (out_key, raw_slot, dil_slot, vis, label)
for _key, (_cimg, _vis, _n, _lab) in AGREEMENT.items():
    _shown = _cimg.updateMask(_cimg.gt(0))
    _rs = _job("agree_" + _key + "::raw", _shown.clip(brazil), _vis)
    _ds = _job("agree_" + _key + "::dil",
               _dilate(_cimg.gte(1)).multiply(_cimg).updateMask(_cimg.gt(0)).clip(brazil), _vis)
    _agreement_specs.append(("agree_" + _key, _rs, _ds, _vis, "%s [%d sources]" % (_lab, _n)))

# (3) Gain layers (after-2020 change), single-colour masks, raw + dilated.
_gain_specs = []  # (out_key, hex, raw_slot, dil_slot, label)
for _key, (_mask, _hex, _lab) in GAIN.items():
    _vis = {"palette": [_hex]}
    _m01 = ee.Image(_mask).gt(0)
    _rs = _job("gain_" + _key + "::raw", _m01.selfMask().clip(brazil), _vis)
    _ds = _job("gain_" + _key + "::dil", _dilate(_m01).clip(brazil), _vis)
    _gain_specs.append(("gain_" + _key, _hex, _rs, _ds, _lab))

# (3b) Verdict layers (default-on result view): 3-colour verdict with pale/solid confidence baked in.
_PALE = {"low": "a6dcb4", "more": "f8cd8a", "high": "f0a0a0"}
_SOLID = {"low": "41ab5d", "more": "f08c00", "high": "e31a1c"}
_VERDICT6_PALETTE = [_PALE["low"], _SOLID["low"], _PALE["more"], _SOLID["more"], _PALE["high"], _SOLID["high"]]
_VERDICT6_VIS = {"min": 1, "max": 6, "palette": _VERDICT6_PALETTE}


def _verdict6(v3):  # 3-class verdict -> 6-value pale/solid image gated by solid_mask
    low, more, high = v3.eq(1), v3.eq(2), v3.eq(3)
    out = ee.Image(0)
    out = out.where(low, 1).where(low.And(solid_mask), 2)
    out = out.where(more, 3).where(more.And(solid_mask), 4)
    out = out.where(high, 5).where(high.And(solid_mask), 6)
    return out.selfMask()


_verdict_specs = []  # (out_key, raw_slot, dil_slot, label)
for _v3, _key, _tag in [(verdict3_or, "verdict_or", "OR"), (verdict3_conv, "verdict_conv", "k>=2")]:
    _v6 = _verdict6(_v3)
    _rs = _job(_key + "::raw", _v6.clip(brazil), _VERDICT6_VIS)
    _ds = _job(_key + "::dil", _dilate(_v3).clip(brazil), _VERDICT_VIS)
    _verdict_specs.append((_key, _rs, _ds,
                           "final verdict (%s): green=low / amber=more-info / red=high "
                           "(solid >=2 agree, pale single-source or overlap)" % _tag))

# (3c) Overlap toggle: the cross-class 2020 overlap, single grey wash (diagnostic, default off).
_overlap_vis = {"palette": ["6b6b8a"]}
_overlap_raw = _job("overlap2020::raw", overlap2020.selfMask().clip(brazil), _overlap_vis)
_overlap_dil = _job("overlap2020::dil", _dilate(overlap2020).clip(brazil), _overlap_vis)

# (4) Sentinel-2 true-colour backgrounds (before / after).
_s2_specs = [
    ("s2_2020", _job("s2_2020", s2("2020-01-01", "2020-12-31").clip(brazil), _S2_VIS), "Sentinel-2 2020 (before)"),
    ("s2_2425", _job("s2_2425", s2("2024-06-01", "2025-12-31").clip(brazil), _S2_VIS), "Sentinel-2 2024-25 (after)"),
]

# ---- resolve EVERY declared job concurrently --------------------------------
def _resolve_one(slot_id):
    _img, _vis = _jobs[slot_id]
    return slot_id, _xyz(_img, _vis)


print("Resolving %d tile templates concurrently (getMapId, max_workers=16)..." % len(_jobs))
_t0 = _time.time()
with ThreadPoolExecutor(max_workers=16) as _ex:
    _fut_to_slot = {_ex.submit(_resolve_one, sid): sid for sid in _jobs}
    for _fut, _slot in _fut_to_slot.items():
        try:
            _sid, _u = _fut.result()
            _resolved[_sid] = _u
        except Exception as _e:  # noqa: BLE001 -- record per slot, do not let one failure kill the build
            _errors[_slot] = repr(_e)
# A failed resolve leaves its id out of _resolved; surface every such slot + its error (never silent).
if _errors:
    print("WARNING: %d tile template(s) failed to resolve:" % len(_errors))
    for _slot, _msg in _errors.items():
        print("  %s -> %s" % (_slot, _msg))
print("Resolved %d/%d tile templates in %.1fs." % (len(_resolved), len(_jobs), _time.time() - _t0))


def _url(slot_id):  # fetch a resolved url; raise a clear error if a needed slot failed (not silent)
    if slot_id not in _resolved:
        raise RuntimeError("tile template '%s' failed to resolve via getMapId (see warnings above)" % slot_id)
    return _resolved[slot_id]


# ---- assemble the registry in the SAME order / keys / structure as the serial build ----
registry = {}
for _key, _hex, _rs, _ds, _label in _pathway_specs:
    registry[_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _hex,
                      "kind": "pathway", "label": _label}
for _out_key, _rs, _ds, _vis, _label in _agreement_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _vis["palette"][-1],
                          "kind": "agreement", "label": _label, "vis": _vis}
for _out_key, _hex, _rs, _ds, _label in _gain_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _hex,
                          "kind": "gain", "label": _label}
for _out_key, _rs, _ds, _label in _verdict_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _SOLID["high"],
                          "kind": "verdict", "label": _label}
registry["overlap2020"] = {
    "raw_url": _url(_overlap_raw), "dilated_url": _url(_overlap_dil), "colour": "6b6b8a", "kind": "overlap",
    "label": "cross-class 2020 overlap (pixel matched >1 of: ag / regen / plantation / primary / other-land; resolved by precedence)",
}

_STATS = {}  # build-time stats panel removed (the area sum hit the Image.reproject limit)

S2_BACKGROUNDS = {_k: (_url(_slot), _lab) for _k, _slot, _lab in _s2_specs}

print("Resolved %d overlay layers + %d S2 backgrounds." % (len(registry), len(S2_BACKGROUNDS)))


## Assemble the interactive HTML (Mermaid tree + Leaflet map)

This builds the Mermaid decision-tree source and the Leaflet + Mermaid HTML scaffold, then injects the resolved tile URLs and node/terminal mappings into it.

In [ ]:
# ============================================================================
# Mermaid decision-tree source + node/terminal mappings + the Leaflet+Mermaid
# HTML scaffold. The resolved tile URLs (registry / S2_BACKGROUNDS) are injected
# into the scaffold as a JSON CONFIG object.
# ============================================================================

# NODE -> layer(s): a single layer key (string) or a list (a small dropdown sub-group).
NODE_TO_LAYER = {
    "OL2020": "agree_other_land",
    "AG2020": ["agree_ag_whole", "agree_cropland", "agree_treecrop", "agree_pasture"],
    "PL2020": "agree_plantation",
    "RP2020": "agree_regen",
    "PR2020": "agree_primary",
    "AG2025": ["gain_ag_whole_gain", "gain_cropland_gain", "gain_treecrop_gain", "gain_pasture_gain"],
    "pl_PL2025": "gain_plantation_presence",
    "rp_PL2025": "gain_plantation_gain",
    "pr_PL2025": "gain_plantation_gain",
    "rp_PR2025": "gain_forest_present_2025",
    "rp_RP2025": "gain_regen_stayed_2025",
    "pr_PR2025": "gain_still_primary_2025",
    "pl_OL2025": "gain_other_land_gain",
    "rp_OL2025": "gain_other_land_gain",
    "pr_OL2025": "gain_other_land_gain",
    "ol_OL2025": "gain_other_land_present_2025",
}
AGREEMENT_OPTION_LABELS = {
    "agree_ag_whole": "agriculture as a whole (distinct sources)",
    "agree_cropland": "cropland (annual/temporary)",
    "agree_treecrop": "tree-crop (perennial)",
    "agree_pasture": "pasture",
    "gain_cropland_gain": "cropland gain (ESRI+GLAD)",
    "gain_treecrop_gain": "tree-crop gain (FDaP)",
    "gain_pasture_gain": "pasture gain (MapBiomas+GPW)",
    "gain_ag_whole_gain": "agriculture as a whole",
}
TERMINAL_TO_CODE = {
    "low_ol2020": (11, "low"),
    "more_ol2020_changed": (13, "more"),
    "low_ag2020": (1, "low"),
    "high_defor": (6, "high"),
    "more_nonforest": (5, "more"),
    "low_pl_stable": (4, "low"),
    "low_pl_other": (18, "low"),
    "more_pl": (8, "more"),
    "low_rp_primary": (3, "low"),
    "low_rp_stay": (3, "low"),
    "high_degrad": (7, "high"),
    "low_rp_other": (17, "low"),
    "more_rp": (10, "more"),
    "low_pr_primary": (2, "low"),
    "low_pr_other": (16, "low"),
    "high_degrad_pr": (12, "high"),
    "more_pr": (9, "more"),
}
CLASS_CODES = {"low": LOW_CODES, "high": HIGH_CODES, "more": MORE_CODES}

CODE_TO_TERMINALS = {}
for _tid, (_c, _v) in TERMINAL_TO_CODE.items():
    CODE_TO_TERMINALS.setdefault(str(_c), []).append(_tid)

CODE_LABELS = {str(c): _CODE_NAMES[c] for c in range(1, 19)}

BOOKMARKS = [
    ["Tocantinzinho post-2020 mine", -6.0519, -56.2935, 13],
    ["Pasture expansion (Amazon basin)", -6.4478, -56.1887, 13],
    ["Southern silviculture belt", -22.0, -48.0, 7],
    ["Cerrado pasture-gain hotspot (Matopiba)", -9.5, -45.5, 8],
]

# --- Mermaid timber decision-tree source -------------------------------------
TIMBER_NODES = {
    "OL2020": "Other land use 2020?", "AG2020": "Agriculture in 2020?",
    "AG2025": "Agriculture after 2020?<br/><small>(deforestation, gated on treecover 2020)</small>",
    "PL2020": "Plantation in 2020?", "RP2020": "Regenerating-planted forest 2020?",
    "PR2020": "Primary forest 2020?",
    "ol_OL2025": "Other land use 2025?",
    "pl_PL2025": "Still plantation? (presence)", "pl_OL2025": "Became other land? (gain)",
    "rp_PR2025": "Primary forest 2025?", "rp_RP2025": "Regenerating-planted forest 2025?",
    "rp_PL2025": "New plantation? (degradation)", "rp_OL2025": "Became other land? (gain)",
    "pr_PR2025": "Primary forest 2025?", "pr_OL2025": "Became other land? (gain)",
    "pr_PL2025": "New plantation? (degradation)",
}
TIMBER_NODE_KIND = {
    "OL2020": "landuse", "AG2020": "landuse", "AG2025": "landuse",
    "PL2020": "forest", "RP2020": "forest", "PR2020": "forest",
    "ol_OL2025": "landuse",
    "pl_PL2025": "forest", "pl_OL2025": "landuse", "rp_PR2025": "forest",
    "rp_RP2025": "forest", "rp_PL2025": "forest", "rp_OL2025": "landuse",
    "pr_PR2025": "forest", "pr_OL2025": "landuse", "pr_PL2025": "forest",
}
TIMBER_TERMINALS = {
    "low_ol2020": "low", "more_ol2020_changed": "more", "low_ag2020": "low", "more_nonforest": "more",
    "low_pl_stable": "low", "low_pl_other": "low", "low_rp_primary": "low",
    "low_rp_stay": "low", "low_rp_other": "low", "low_pr_primary": "low",
    "low_pr_other": "low", "more_rp": "more", "more_pr": "more", "more_pl": "more",
    "high_defor": "high", "high_degrad": "high", "high_degrad_pr": "high",
}
TIMBER_EDGES = """
  OL2020 -- No --> AG2020
  AG2020 -- No --> AG2025
  AG2025 -- No --> PL2020
  PL2020 -- No --> RP2020
  RP2020 -- No --> PR2020
  OL2020 -- Yes --> ol_OL2025
  ol_OL2025 -- Yes --> low_ol2020
  ol_OL2025 -- No --> more_ol2020_changed
  AG2020 -- Yes --> low_ag2020
  AG2025 -- "Yes (had treecover 2020)" --> high_defor
  PR2020 -- No --> more_nonforest
  PL2020 -- Yes --> pl_PL2025
  pl_PL2025 -- Yes --> low_pl_stable
  pl_PL2025 -- No --> pl_OL2025
  pl_OL2025 -- Yes --> low_pl_other
  pl_OL2025 -- No --> more_pl
  RP2020 -- Yes --> rp_PR2025
  rp_PR2025 -- Yes --> low_rp_primary
  rp_PR2025 -- No --> rp_RP2025
  rp_RP2025 -- Yes --> low_rp_stay
  rp_RP2025 -- No --> rp_PL2025
  rp_PL2025 -- Yes --> high_degrad
  rp_PL2025 -- No --> rp_OL2025
  rp_OL2025 -- Yes --> low_rp_other
  rp_OL2025 -- No --> more_rp
  PR2020 -- Yes --> pr_PR2025
  pr_PR2025 -- Yes --> low_pr_primary
  pr_PR2025 -- No --> pr_OL2025
  pr_OL2025 -- Yes --> low_pr_other
  pr_OL2025 -- No --> pr_PL2025
  pr_PL2025 -- Yes --> high_degrad_pr
  pr_PL2025 -- No --> more_pr
"""
_TERMTEXT = {"low": "Low risk", "more": "More info needed", "high": "High risk"}
_TERM_CAPTION = {
    "low_ol2020": "Low risk<br/><small>other land 2020 (stable)</small>",
    "more_ol2020_changed": "More info needed<br/><small>other land 2020 changed</small>",
    "low_pl_stable": "Low risk<br/><small>stable plantation (presence)</small>",
    "low_pl_other": "Low risk<br/><small>plantation 2020 -&gt; other land</small>",
    "low_rp_stay": "Low risk<br/><small>regen stayed forest</small>",
    "low_rp_other": "Low risk<br/><small>regen 2020 -&gt; other land</small>",
    "low_pr_primary": "Low risk<br/><small>still primary</small>",
    "low_pr_other": "Low risk<br/><small>primary 2020 -&gt; other land</small>",
}
_CLASSDEFS = """
  classDef forest fill:#bbf7d0,stroke:#16a34a,color:#064e3b;
  classDef landuse fill:#fff7ed,stroke:#f97316,color:#7c2d12;
  classDef low fill:#16a34a,color:#fff,stroke:#166534;
  classDef more fill:#f08c00,color:#fff,stroke:#c56f00;
  classDef high fill:#dc2626,color:#fff,stroke:#b91c1c;
  classDef q2020 stroke-width:3px;
  classDef q2025 stroke-width:3px,stroke-dasharray:4 3;
"""
_Q2020 = {n for n in NODE_TO_LAYER if not n.endswith("2025") and "2025" not in n}
_Q2025 = {n for n in NODE_TO_LAYER if n.endswith("2025") or "2025" in n}


def build_mermaid_source():
    s = "flowchart TB\n"
    for nid, label in TIMBER_NODES.items():
        s += '  %s{"%s"}\n' % (nid, label)
    for tid, kind in TIMBER_TERMINALS.items():
        base_label = _TERM_CAPTION.get(tid, _TERMTEXT[kind])
        code_val = TERMINAL_TO_CODE.get(tid, (None,))[0]
        label = base_label + ("<br/><small>Code: %d</small>" % code_val if code_val is not None else "")
        s += '  %s(["%s"])\n' % (tid, label)
    s += TIMBER_EDGES + _CLASSDEFS
    for nid, kind in TIMBER_NODE_KIND.items():
        s += "  class %s %s\n" % (nid, kind)
    for nid in _Q2020:
        s += "  class %s q2020\n" % nid
    for nid in _Q2025:
        s += "  class %s q2025\n" % nid
    for tid, kind in TIMBER_TERMINALS.items():
        s += "  class %s %s\n" % (tid, kind)
    return s


MERMAID_SRC = build_mermaid_source()

_JS_CONFIG = {
    "registry": registry,
    "s2Backgrounds": {k: {"url": u, "label": lab} for k, (u, lab) in S2_BACKGROUNDS.items()},
    "nodeToLayer": NODE_TO_LAYER,
    "agreementOptionLabels": AGREEMENT_OPTION_LABELS,
    "terminalToCode": {k: {"code": v[0], "verdict": v[1]} for k, v in TERMINAL_TO_CODE.items()},
    "codeToTerminals": CODE_TO_TERMINALS,
    "classCodes": CLASS_CODES,
    "codeLabels": CODE_LABELS,
    "bookmarks": BOOKMARKS,
    "palette": _PALETTE_BY_CODE,
    "allNodes": list(TIMBER_NODES.keys()),
    "allTerminals": list(TIMBER_TERMINALS.keys()),
    "verdictLayers": {"or": "verdict_or", "conv": "verdict_conv"},
    "overlapLayer": "overlap2020",
    "stats": _STATS,
    "verdictSwatch": {"low": _SOLID["low"], "more": _SOLID["more"], "high": _SOLID["high"]},
}

# --- the Leaflet + Mermaid HTML scaffold -------------------------------------
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>WHISP timber: interactive decision-tree / map viewer</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<!-- Leaflet.draw: the polygon-draw control for the "draw -> whisp -> colour by risk" feature. -->
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css" />
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<style>
  html, body { height: 100%; }
  body { font-family: -apple-system, Segoe UI, Helvetica, Arial, sans-serif; margin: 0; color: #1f2937; }
  header { padding: .7rem 1.1rem; background: #166534; color: #fff; }
  header h1 { font-size: 1.05rem; margin: 0; }
  header p { margin: .25rem 0 0; font-size: .78rem; opacity: .92; }
  .wrap { display: flex; height: calc(100vh - 64px); min-height: 760px; }
  #treepane { width: 46%; min-width: 200px; min-height: 760px; overflow: auto; border-right: 1px solid #e5e7eb; padding: .8rem; background: #fafafa; flex: 0 0 auto; }
  #splitter { flex: 0 0 6px; cursor: col-resize; background: #e5e7eb; border-left: 1px solid #d1d5db;
              border-right: 1px solid #d1d5db; }
  #splitter:hover { background: #cbd5e1; }
  body.dragging { cursor: col-resize; user-select: none; }
  #treepane svg { max-width: 100%; height: auto; }
  .node-clickable { cursor: pointer; }
  g.node-active rect, g.node-active polygon, g.node-active circle, g.node-active path {
    stroke: #ff6d00 !important; stroke-width: 5px !important;
    filter: drop-shadow(0 0 6px #ff6d00) drop-shadow(0 0 3px #1d4ed8);
  }
  g.node-active { filter: drop-shadow(0 0 8px rgba(255,109,0,.9)); }
  #mappane { flex: 1; display: flex; flex-direction: column; position: relative; }
  #controls { padding: .5rem .8rem; background: #f3f4f6; border-bottom: 1px solid #e5e7eb; font-size: .8rem; }
  #controls > * { margin-right: .8rem; vertical-align: middle; }
  #controls select, #controls button { font-size: .8rem; padding: .2rem .35rem; }
  #controls .bm button { margin-left: .25rem; }
  #map { flex: 1; min-height: 700px; height: 700px; }
  #status { font-size: .78rem; padding: .35rem .8rem; background: #fff; border-bottom: 1px solid #e5e7eb; min-height: 1.1rem; }
  #legend { position: absolute; bottom: 16px; right: 12px; z-index: 1000; background: #fff;
            padding: 6px 9px; font-size: 11px; border: 1px solid #999; border-radius: 5px;
            box-shadow: 0 1px 4px rgba(0,0,0,.3); max-width: 250px; }
  #legend .sw { display: inline-block; width: 12px; height: 12px; border: 1px solid #777;
                margin-right: 5px; vertical-align: middle; }
  .hint { color: #6b7280; }
</style>
</head>
<body>
<header>
  <h1>WHISP timber: decision tree linked to the map</h1>
  <p>Click a <b>2020 question</b> (solid thick border) to map that class's 2020 source-agreement layer,
     or a <b>2025 question</b> (dashed thick border) for the after-2020 gain/change layer.
     Click a <b>verdict</b> to pick a feeding pathway code and map it. The <b>agreement</b> toggle swaps
     the pathway set between OR (any 1 product, the default) and k&gt;=2 (at least 2 agreeing products).
     Pick a <b>base</b> (a neutral grey OSM that makes the coloured overlays pop, or Esri imagery). The two
     <b>S2</b> backgrounds are independent toggles; tick either or both to bring the 2020 / 2024-25 imagery
     in beneath the overlays.
     EUDR is one example compliance framing; this is a general deforestation-risk view.</p>
</header>

<div class="wrap">
  <div id="treepane">rendering tree...</div>
  <div id="splitter" title="drag to resize the tree / map panes"></div>
  <div id="mappane">
    <div id="controls">
      <span><b id="curlabel">no layer</b></span>
      <button id="verdictBtn" title="show the default 3-colour final-verdict result map">final verdict</button>
      <label class="hint">layer:
        <select id="codeSelect" disabled></select></label>
      <label class="hint">agreement:
        <select id="verdictSelect" title="Experimental. k>=2 requires >=2 agreeing products, but only on the primary-2020 and deforestation nodes; all other nodes stay OR and single-source pixels drop out.">
          <option value="or" selected>OR (any 1 product)</option>
          <option value="conv">k&gt;=2 agreement (experimental)</option>
        </select></label>
      <label><input type="checkbox" id="dilateChk"> dilate (zoom-stable)</label>
      <label><input type="checkbox" id="mainLayerChk" checked> show layer</label>
      <label title="cross-class 2020 overlap: a pixel matched more than one 2020 class (resolved by precedence)"><input type="checkbox" id="overlapChk"> overlap</label>
      <span class="hint">base:
        <label><input type="radio" name="base" value="positron" checked> grey OSM</label>
        <label><input type="radio" name="base" value="esri"> Esri</label></span>
      <span class="hint">S2:
        <label><input type="checkbox" id="s2_2020chk"> 2020</label>
        <label><input type="checkbox" id="s2_2425chk"> 2024-25</label></span>
      <span class="bm hint">go to:<span id="bookmarks"></span></span>
      <button id="clearBtn">clear</button>
    </div>
    <div id="status" class="hint">Final verdict shown (green=low / amber=more-info / red=high). Click a node or a verdict for the diagnostic drill-down.</div>
    <div id="map"></div>
    <div id="legend"><b>Legend</b><div id="legendBody" class="hint">select a layer</div></div>
  </div>
</div>

<script type="module">
import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
mermaid.initialize({ startOnLoad: false, securityLevel: 'loose', flowchart: { htmlLabels: true, rankSpacing: 80 } });

const CONFIG = __CONFIG__;
const MERMAID_SRC = __MERMAID__;

const map = L.map('map', { center: [-14, -50], zoom: 4 });
map.createPane('s2lo');  map.getPane('s2lo').style.zIndex  = 240;
map.createPane('s2hi');  map.getPane('s2hi').style.zIndex  = 260;
map.createPane('ovpane'); map.getPane('ovpane').style.zIndex = 450;

const BASES = {
  esri: L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    { maxZoom: 19, attribution: 'Esri World Imagery' }),
  positron: L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png',
    { maxZoom: 19, subdomains: 'abcd', attribution: '&copy; OpenStreetMap, &copy; CARTO' }),
};
let currentBase = 'positron';
BASES.positron.addTo(map);
function setBase(value) {
  if (value === currentBase || !BASES[value]) return;
  map.removeLayer(BASES[currentBase]);
  BASES[value].addTo(map);
  BASES[value].bringToBack();
  currentBase = value;
}
document.querySelectorAll('input[name="base"]').forEach(r =>
  r.addEventListener('change', e => { if (e.target.checked) setBase(e.target.value); }));

const S2 = {
  s2_2020: L.tileLayer(CONFIG.s2Backgrounds.s2_2020.url, { pane: 's2lo', maxZoom: 19, attribution: CONFIG.s2Backgrounds.s2_2020.label }),
  s2_2425: L.tileLayer(CONFIG.s2Backgrounds.s2_2425.url, { pane: 's2hi', maxZoom: 19, attribution: CONFIG.s2Backgrounds.s2_2425.label }),
};
document.getElementById('s2_2020chk').addEventListener('change', e => {
  if (e.target.checked) S2.s2_2020.addTo(map); else map.removeLayer(S2.s2_2020);
});
document.getElementById('s2_2425chk').addEventListener('change', e => {
  if (e.target.checked) S2.s2_2425.addTo(map); else map.removeLayer(S2.s2_2425);
});

const LAYERS = {};
for (const [key, meta] of Object.entries(CONFIG.registry)) {
  LAYERS[key] = {
    raw: L.tileLayer(meta.raw_url, { pane: 'ovpane', opacity: 0.85, maxZoom: 19 }),
    dilated: L.tileLayer(meta.dilated_url, { pane: 'ovpane', opacity: 0.85, maxZoom: 19 }),
    meta: meta,
  };
}

let verdictMode = 'or';
function pathKey(code) { return (verdictMode === 'conv' ? 'pathconv_' : 'pathor_') + code; }
function verdictKey() { return CONFIG.verdictLayers[verdictMode]; }

const INDEPENDENT_KEYS = [CONFIG.overlapLayer];

let currentKey = null;
function dilateOn() { return document.getElementById('dilateChk').checked; }
function mainLayerOn() { return document.getElementById('mainLayerChk').checked; }

function hideAll() {
  for (const k in LAYERS) {
    if (INDEPENDENT_KEYS.includes(k)) continue;
    map.removeLayer(LAYERS[k].raw); map.removeLayer(LAYERS[k].dilated);
  }
}
function showLayer(key) {
  hideAll();
  currentKey = key;
  if (!key) { setStatus('No layer shown. Click a node or a verdict.'); setLegend(null); return; }
  // FIX: guard a missing/renamed registry key. If the layer does not exist (e.g. a verdict key that
  // drifted out of CONFIG.verdictLayers / the registry), surface it instead of dereferencing undefined and
  // throwing, which previously aborted showLayer silently and left the map un-repainted.
  const lyr = LAYERS[key];
  if (!lyr) {
    currentKey = null;
    setStatus('layer "' + key + '" is not registered (no tiles to show)');
    setLegend(null);
    console.warn('showLayer: no LAYERS entry for key', key, '- known keys:', Object.keys(LAYERS));
    return;
  }
  if (mainLayerOn()) {
    const tl = (dilateOn() ? lyr.dilated : lyr.raw);
    tl.addTo(map);
    // FIX: force a tile refresh on (re-)add. Re-adding the SAME cached Leaflet tile-layer object (the
    // verdict layer is re-added every time the "final verdict" button is pressed after navigating away)
    // could leave Leaflet serving its already-loaded tile set without re-requesting for the current view,
    // so the map appeared NOT to repaint. redraw() re-fetches the tiles for the active viewport.
    if (typeof tl.redraw === 'function') tl.redraw();
  }
  document.getElementById('curlabel').textContent = lyr.meta.label;
  setStatus(lyr.meta.label + (dilateOn() ? '  (dilated)' : '  (raw 30 m)'));
  setLegend(key);
}
function showVerdict() {
  dropdownMode = null;
  const sel = document.getElementById('codeSelect'); sel.disabled = true; sel.innerHTML = '';
  showLayer(verdictKey());
}
function setStatus(t) { document.getElementById('status').textContent = t; }

function setLegend(key) {
  const body = document.getElementById('legendBody');
  if (!key) { body.innerHTML = 'select a layer'; return; }
  const meta = LAYERS[key].meta;
  if (meta.kind === 'verdict') {
    const sw = CONFIG.verdictSwatch;
    body.innerHTML = "<div style='margin-bottom:3px'>final verdict (" + verdictMode.toUpperCase() + ")</div>" +
      "<div><span class='sw' style='background:#" + sw.low + "'></span>low</div>" +
      "<div><span class='sw' style='background:#" + sw.more + "'></span>more info</div>" +
      "<div><span class='sw' style='background:#" + sw.high + "'></span>high</div>" +
      "<div class='hint' style='margin-top:3px;font-size:10px'>solid = &gt;=2 products agree; pale = single-source or cross-class overlap</div>";
  } else if (meta.kind === 'agreement') {
    const pal = meta.vis.palette;
    body.innerHTML = "<div style='margin-bottom:3px'>" + meta.label + "</div>" +
      pal.map((c, i) => "<div><span class='sw' style='background:#" + c + "'></span>k&gt;=" + (i + 1) + "</div>").join('');
  } else {
    body.innerHTML = "<div><span class='sw' style='background:#" + meta.colour + "'></span>" + meta.label + "</div>";
  }
}

function renderStats() { /* stats panel removed */ }

document.getElementById('dilateChk').addEventListener('change', () => { if (currentKey) showLayer(currentKey); });
document.getElementById('mainLayerChk').addEventListener('change', () => { if (currentKey) showLayer(currentKey); });
document.getElementById('clearBtn').addEventListener('click', () => {
  showLayer(null);
  const sel = document.getElementById('codeSelect'); sel.disabled = true; sel.innerHTML = ''; dropdownMode = null;
  document.getElementById('curlabel').textContent = 'no layer';
  if (typeof _activeNode !== 'undefined' && _activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
  if (typeof clearActiveSet === 'function') clearActiveSet();
});

const codeSelect = document.getElementById('codeSelect');
let dropdownMode = null;
function populateCodes(codes, selectedCode) {
  codeSelect.innerHTML = '';
  codes.forEach(c => {
    const opt = document.createElement('option');
    opt.value = String(c);
    opt.textContent = 'code ' + c + ': ' + CONFIG.codeLabels[String(c)];
    if (c === selectedCode) opt.selected = true;
    codeSelect.appendChild(opt);
  });
  codeSelect.disabled = false; dropdownMode = 'pathway';
}
function populateSubgroup(layerKeys, selectedKey) {
  codeSelect.innerHTML = '';
  layerKeys.forEach(k => {
    const opt = document.createElement('option');
    opt.value = k;
    opt.textContent = CONFIG.agreementOptionLabels[k] || (LAYERS[k] ? LAYERS[k].meta.label : k);
    if (k === selectedKey) opt.selected = true;
    codeSelect.appendChild(opt);
  });
  codeSelect.disabled = false; dropdownMode = 'subgroup';
}
codeSelect.addEventListener('change', () => {
  if (dropdownMode === 'pathway') {
    showLayer(pathKey(codeSelect.value));
    highlightTerminals(CONFIG.codeToTerminals[String(codeSelect.value)]);
  } else {
    showLayer(codeSelect.value);
  }
});

document.getElementById('verdictSelect').addEventListener('change', e => {
  const showingVerdict = (currentKey === CONFIG.verdictLayers.or || currentKey === CONFIG.verdictLayers.conv);
  verdictMode = e.target.value;
  renderStats();
  if (dropdownMode === 'pathway' && codeSelect.value) {
    showLayer(pathKey(codeSelect.value));
  } else if (showingVerdict) {
    showVerdict();
  }
});

function onNodeClick(nodeId) {
  const mapped = CONFIG.nodeToLayer[nodeId];
  if (!mapped) {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    setStatus(nodeId + ': no dedicated layer');
    return;
  }
  if (Array.isArray(mapped)) {
    populateSubgroup(mapped, mapped[0]);
    showLayer(mapped[0]);
  } else {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    showLayer(mapped);
  }
}
function onTerminalClick(termId) {
  const info = CONFIG.terminalToCode[termId];
  if (!info) {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    setStatus(termId + ': no dedicated layer');
    return;
  }
  const codes = CONFIG.classCodes[info.verdict];
  populateCodes(codes, info.code);
  showLayer(pathKey(info.code));
}

let _activeNode = null;
const NODE_G = {};
let _activeSet = [];
function clearActiveSet() {
  _activeSet.forEach(g => { try { g.classList.remove('node-active'); } catch (e) { } });
  _activeSet = [];
}
function highlightTerminals(termIds) {
  try {
    if (_activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
    clearActiveSet();
    (termIds || []).forEach(tid => {
      const g = NODE_G[tid];
      if (g) { g.classList.add('node-active'); _activeSet.push(g); }
    });
    if (_activeSet.length) scrollTerminalIntoTreeView(_activeSet[0]);
  } catch (err) { }
}
function scrollTerminalIntoTreeView(g) {
  try {
    const pane = document.getElementById('treepane');
    const pr = pane.getBoundingClientRect();
    const gr = g.getBoundingClientRect();
    const fullyVisible = gr.top >= pr.top && gr.bottom <= pr.bottom &&
                         gr.left >= pr.left && gr.right <= pr.right;
    if (fullyVisible) return;
    if (typeof g.scrollIntoView === 'function') {
      g.scrollIntoView({ behavior: 'smooth', block: 'center', inline: 'center' });
    } else {
      pane.scrollTop += (gr.top - pr.top) - (pr.height - gr.height) / 2;
      pane.scrollLeft += (gr.left - pr.left) - (pr.width - gr.width) / 2;
    }
  } catch (e) { }
}
function setActiveNode(fromEl, boundG) {
  try {
    clearActiveSet();
    let el = fromEl;
    while (el && !(el.classList && el.classList.contains('node')) && el !== document) {
      el = el.parentNode;
    }
    const target = (el && el.classList && el.classList.contains('node')) ? el : boundG;
    if (!target) return;
    if (_activeNode && _activeNode !== target) _activeNode.classList.remove('node-active');
    target.classList.add('node-active');
    _activeNode = target;
  } catch (err) {
    try {
      if (boundG) {
        if (_activeNode && _activeNode !== boundG) _activeNode.classList.remove('node-active');
        boundG.classList.add('node-active');
        _activeNode = boundG;
      }
    } catch (e2) { }
  }
}

async function renderTree() {
  const { svg } = await mermaid.render('timberSvg', MERMAID_SRC);
  document.getElementById('treepane').innerHTML = svg;
  const groups = document.querySelectorAll('#treepane svg g.node, #treepane svg g[id]');
  groups.forEach(g => {
    const gid = g.id || '';
    for (const nodeId of CONFIG.allNodes) {
      if (gid.includes('-' + nodeId + '-') || gid === ('flowchart-' + nodeId)) {
        g.classList.add('node-clickable');
        NODE_G[nodeId] = g;
        g.addEventListener('click', (ev) => { onNodeClick(nodeId); setActiveNode(ev.target, g); });
      }
    }
    for (const termId of CONFIG.allTerminals) {
      if (gid.includes('-' + termId + '-') || gid === ('flowchart-' + termId)) {
        g.classList.add('node-clickable');
        NODE_G[termId] = g;
        g.addEventListener('click', (ev) => { onTerminalClick(termId); setActiveNode(ev.target, g); });
      }
    }
  });
  setTimeout(() => map.invalidateSize(), 200);
}
renderTree();
renderStats();
showVerdict();

document.getElementById('verdictBtn').addEventListener('click', () => {
  if (typeof _activeNode !== 'undefined' && _activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
  if (typeof clearActiveSet === 'function') clearActiveSet();
  showVerdict();
});

function toggleIndependent(checked, key) {
  const lyr = LAYERS[key];
  map.removeLayer(lyr.raw); map.removeLayer(lyr.dilated);
  if (checked) (dilateOn() ? lyr.dilated : lyr.raw).addTo(map);
}
document.getElementById('overlapChk').addEventListener('change', e =>
  toggleIndependent(e.target.checked, CONFIG.overlapLayer));
document.getElementById('dilateChk').addEventListener('change', () => {
  toggleIndependent(document.getElementById('overlapChk').checked, CONFIG.overlapLayer);
});

const bm = document.getElementById('bookmarks');
CONFIG.bookmarks.forEach(([name, lat, lon, z]) => {
  const b = document.createElement('button');
  b.textContent = name;
  b.title = name + ' (' + lat + ', ' + lon + ')';
  b.addEventListener('click', () => map.setView([lat, lon], z));
  bm.appendChild(b);
});

// =====================================================================================================
// DRAW A POLYGON -> RUN WHISP -> ATTACH ind_/risk_ COLUMNS + COLOUR THE POLYGON BY RISK
// -----------------------------------------------------------------------------------------------------
// Out of the box this targets the LIVE public Whisp API (set WHISP_API_KEY and it works). To use a dev
// instance instead, change WHISP_API_BASE (one line). The DRAW + PARSE + COLOUR logic works regardless;
// only the submit/poll network path needs a reachable API + key.
// CONFIG (edit these):
const WHISP_API_BASE = "https://whisp.openforis.org/api";  // live public API (routes under /api). For the
                                                           // dev instance, change THIS line to the dev base.
const WHISP_API_KEY  = "";                                 // <-- put your X-API-KEY here (blank = draw only, no analysis)
const RISK_COLUMN    = "risk_timber";                      // verdict column (low/high/more) , the fallback colour
const PATHWAY_COLUMN = "risk_timber_pathway";              // the SPECIFIC pathway label -> the exact map code colour

// risk_timber (low/high/more-info) -> the viewer's verdict map palette (the FALLBACK if the specific
// pathway label is unknown). Same hexes the map uses: low = green 41ab5d, more = amber f08c00, high = red e31a1c.
const RISK_COLOURS = { low: "#41ab5d", more: "#f08c00", high: "#e31a1c", unknown: "#6b7280" };
function riskToColour(val) {
  const v = String(val == null ? "" : val).toLowerCase();
  if (v.indexOf("high") >= 0) return RISK_COLOURS.high;
  if (v.indexOf("low") >= 0) return RISK_COLOURS.low;
  if (v.indexOf("more") >= 0 || v.indexOf("info") >= 0) return RISK_COLOURS.more;
  return RISK_COLOURS.unknown;
}

// risk_timber_pathway label (from add_risk_timber_col in src/openforis_whisp/risk.py) -> the viewer's
// pathway CODE. The code's colour comes from CONFIG.palette (the SAME _PALETTE_BY_CODE the map uses), so a
// drawn polygon gets the EXACT colour the map paints for that pathway. Codes/colours: see PALETTE in the
// build script. Code 14 is shared by the three "<class> 2020 -> other land" labels (as in the map).
const PATHWAY_LABEL_TO_CODE = {
  "low: other land 2020 (stable)": 11,
  "more-info: other land 2020 changed": 13,
  "low: agriculture 2020": 1,
  "high: deforestation": 6,
  "low: stable plantation": 4,
  "low: plantation 2020 -> other land": 14,
  "more-info: plantation 2020, no 2025 state": 8,
  "low: regen matured to primary": 15,
  "low: regen stayed forest": 3,
  "high: regen->plantation degradation": 7,
  "low: regen 2020 -> other land": 14,
  "more-info: regen 2020, no 2025 state": 10,
  "low: still primary": 2,
  "low: primary 2020 -> other land": 14,
  "high: primary->plantation degradation": 12,
  "more-info: primary 2020, no 2025 state": 9,
  "more-info: 2020 state unknown": 5,
};
function _normLabel(s) {  // tolerant match: lowercase, collapse whitespace, normalise the arrow + dashes
  return String(s == null ? "" : s).toLowerCase().trim()
    .replace(/\s+/g, " ").replace(/-+>/g, "->").replace(/→/g, "->");
}
const _PATHWAY_LOOKUP = {};
for (const k of Object.keys(PATHWAY_LABEL_TO_CODE)) _PATHWAY_LOOKUP[_normLabel(k)] = PATHWAY_LABEL_TO_CODE[k];
// Resolve a result's properties -> { code, colour, verdict } using the specific pathway label first,
// falling back to the plain risk_timber verdict colour if the label is missing / unrecognised.
function resolvePathwayStyle(props) {
  const label = props ? props[PATHWAY_COLUMN] : null;
  const code = label != null ? _PATHWAY_LOOKUP[_normLabel(label)] : undefined;
  const verdict = props ? props[RISK_COLUMN] : null;
  if (code !== undefined && CONFIG.palette && CONFIG.palette[code]) {
    return { code: code, colour: "#" + CONFIG.palette[code], verdict: verdict, label: label };
  }
  return { code: null, colour: riskToColour(verdict), verdict: verdict, label: label };  // fallback
}

// Drawn polygons live in their own feature group, above the overlays.
map.createPane('drawpane'); map.getPane('drawpane').style.zIndex = 650;
const drawnItems = new L.FeatureGroup();
map.addLayer(drawnItems);
const drawControl = new L.Control.Draw({
  edit: { featureGroup: drawnItems, edit: false, remove: true },
  draw: { polygon: { allowIntersection: false, showArea: true }, polyline: false, rectangle: false,
          circle: false, circlemarker: false, marker: false },
});
map.addControl(drawControl);

function _drawStatus(msg) { setStatus(msg); }

// Collect every ind_/Ind_ or risk_ property (case-insensitive) from a result feature's properties.
function collectIndRisk(props) {
  const out = {};
  if (!props) return out;
  for (const k of Object.keys(props)) {
    const lk = k.toLowerCase();
    if (lk.startsWith("ind_") || lk.startsWith("risk_")) out[k] = props[k];
  }
  return out;
}

// Style a drawn polygon by its SPECIFIC pathway code colour (fallback: the plain risk_timber colour).
function styleByRisk(layer, props) {
  const colour = props ? resolvePathwayStyle(props).colour : RISK_COLOURS.unknown;
  layer.setStyle({ color: colour, weight: 2, fillColor: colour, fillOpacity: 0.45 });
}

// Build a popup: a header line "code N: <pathway label> (LOW/HIGH/MORE-INFO)" then the ind_/risk_ values.
function indRiskPopupHtml(indRisk, style) {
  const verdict = style && style.verdict != null ? String(style.verdict).replace(/_/g, " ").toUpperCase() : "";
  let header = "<b>Whisp result</b>";
  if (style) {
    const codeTxt = style.code != null ? ("code " + style.code + ": ") : "";
    const labelTxt = style.label != null ? style.label : (style.verdict != null ? style.verdict : "n/a");
    header += "<div style='margin-top:2px'><span style='display:inline-block;width:11px;height:11px;"
      + "border:1px solid #777;background:" + style.colour + ";margin-right:5px;vertical-align:middle'></span>"
      + codeTxt + labelTxt + (verdict ? " (" + verdict + ")" : "") + "</div>";
  }
  const keys = Object.keys(indRisk);
  if (!keys.length) return header + "<br/><span style='font-size:11px'>(no ind_/risk_ columns found)</span>";
  const riskFirst = keys.sort((p, q) => {
    const rp = p.toLowerCase().startsWith("risk_") ? 0 : 1;
    const rq = q.toLowerCase().startsWith("risk_") ? 0 : 1;
    return rp - rq || p.localeCompare(q);
  });
  let html = header + "<div style='max-height:200px;overflow:auto;font-size:11px;margin-top:4px'>";
  for (const k of riskFirst) html += "<div><b>" + k + "</b>: " + indRisk[k] + "</div>";
  html += "</div>";
  return html;
}

// Apply a result feature (its properties) to the drawn layer: attach ind_/risk_, colour by pathway, popup.
function applyResultToLayer(layer, resultProps) {
  const indRisk = collectIndRisk(resultProps);
  layer.feature = layer.feature || { type: "Feature", properties: {} };
  layer.feature.properties = Object.assign({}, layer.feature.properties, indRisk);
  const style = resolvePathwayStyle(layer.feature.properties);
  layer.setStyle({ color: style.colour, weight: 2, fillColor: style.colour, fillOpacity: 0.45 });
  layer.bindPopup(indRiskPopupHtml(indRisk, style)).openPopup();
  _drawStatus("Whisp result: " + (style.code != null ? "code " + style.code + " " : "")
    + (style.label != null ? style.label : (style.verdict != null ? style.verdict : "n/a"))
    + "  (" + Object.keys(indRisk).length + " ind_/risk_ columns attached)");
}

// Extract the FIRST feature from a result GeoJSON (FeatureCollection or Feature or array of rows).
function firstResultProps(result) {
  if (!result) return null;
  if (Array.isArray(result) && result.length) return result[0].properties || result[0];
  if (result.type === "FeatureCollection" && result.features && result.features.length)
    return result.features[0].properties || {};
  if (result.type === "Feature") return result.properties || {};
  if (result.features && result.features.length) return result.features[0].properties || {};
  if (result.data) return firstResultProps(result.data);  // unwrap an envelope-in-envelope
  return result.properties || result;  // last resort: treat the object itself as the props bag
}

// Poll GET /status/{token} every ~3s until analysis_completed (or an error / ~2 min timeout).
async function pollStatus(token) {
  const deadline = Date.now() + 120000;  // ~2 min
  while (Date.now() < deadline) {
    await new Promise(r => setTimeout(r, 3000));
    const resp = await fetch(WHISP_API_BASE + "/status/" + token, { headers: { "X-API-KEY": WHISP_API_KEY } });
    const env = await resp.json().catch(() => ({}));
    const code = env && env.code;
    _drawStatus("Whisp analysis: " + (code || "running") + " ...");
    if (code === "analysis_completed") return env;
    if (code && /error|fail/i.test(code)) throw new Error("Whisp analysis returned: " + code + (env.message ? " (" + env.message + ")" : ""));
  }
  throw new Error("Whisp analysis timed out after ~2 minutes.");
}

// Fetch the result GeoJSON for a token: prefer inline data; else GET /generate-geojson/{token} (no key).
async function fetchResultGeojson(env, token) {
  if (env && env.data && (env.data.type || env.data.features || Array.isArray(env.data))) return env.data;
  const resp = await fetch(WHISP_API_BASE + "/generate-geojson/" + token);
  return await resp.json();
}

// Submit a drawn polygon's FeatureCollection to the Whisp API, poll, attach + colour the result.
async function analyzeDrawnLayer(layer, featureCollection) {
  if (!WHISP_API_KEY) {
    layer.bindPopup("Drawn polygon. Set WHISP_API_KEY (top of the script) to analyze it with Whisp.").openPopup();
    _drawStatus("Drawn polygon added. Set WHISP_API_KEY to analyze.");
    return;
  }
  try {
    _drawStatus("Submitting polygon to Whisp ...");
    // Body = the FeatureCollection with an analysisOptions object merged at top level (mirrors
    // whisp-app's useSubmitAnalysis.ts). nationalCodes/generateGeoids are sensible defaults.
    const body = Object.assign({}, featureCollection, {
      analysisOptions: { nationalCodes: [], generateGeoids: false, unitType: "ha" },
    });
    const resp = await fetch(WHISP_API_BASE + "/submit/geojson", {
      method: "POST",
      headers: { "X-API-KEY": WHISP_API_KEY, "Content-Type": "application/json" },
      body: JSON.stringify(body),
    });
    const env = await resp.json().catch(() => ({}));   // envelope: { code, message, data }
    let completedEnv = env;
    let token = (env && env.data && env.data.token) ? env.data.token : (env && env.token);
    if (!env || env.code !== "analysis_completed") {
      if (!token) throw new Error("Whisp submit did not return a token or inline result"
        + (env && env.message ? " (" + env.message + ")" : ""));
      completedEnv = await pollStatus(token);   // poll until analysis_completed / error / timeout
    }
    const result = await fetchResultGeojson(completedEnv, token);
    const props = firstResultProps(result);
    if (!props) throw new Error("Whisp result had no feature properties");
    applyResultToLayer(layer, props);
  } catch (err) {
    _drawStatus("Whisp analysis failed: " + (err && err.message ? err.message : err));
    layer.bindPopup("Whisp analysis failed:<br/>" + (err && err.message ? err.message : err)).openPopup();
  }
}

// draw:created -> add the polygon, build its FeatureCollection, run (or prompt for a key).
map.on(L.Draw.Event.CREATED, function (e) {
  const layer = e.layer;
  drawnItems.addLayer(layer);
  styleByRisk(layer, null);  // neutral until a result comes back
  const feature = layer.toGeoJSON();   // a GeoJSON Feature (Polygon)
  feature.properties = feature.properties || {};
  const featureCollection = { type: "FeatureCollection", features: [feature] };
  analyzeDrawnLayer(layer, featureCollection);
});
map.on(L.Draw.Event.DELETED, function () { _drawStatus("Drawn polygon(s) removed."); });

// ----- FEATURE C: draggable split slider between the tree pane (left) and the map pane (right) -----
(function () {
  const splitter = document.getElementById('splitter');
  const treepane = document.getElementById('treepane');
  const wrap = document.querySelector('.wrap');
  const MIN_TREE = 200;
  const MIN_MAP = 260;
  let dragging = false;
  function onMove(e) {
    if (!dragging) return;
    const rect = wrap.getBoundingClientRect();
    let w = e.clientX - rect.left;
    const maxTree = rect.width - splitter.offsetWidth - MIN_MAP;
    if (w < MIN_TREE) w = MIN_TREE;
    if (w > maxTree) w = maxTree;
    treepane.style.width = w + 'px';
    map.invalidateSize();
  }
  function onUp() {
    if (!dragging) return;
    dragging = false;
    document.body.classList.remove('dragging');
    document.removeEventListener('mousemove', onMove);
    document.removeEventListener('mouseup', onUp);
    map.invalidateSize();
  }
  splitter.addEventListener('mousedown', (e) => {
    e.preventDefault();
    dragging = true;
    document.body.classList.add('dragging');
    document.addEventListener('mousemove', onMove);
    document.addEventListener('mouseup', onUp);
  });
})();
</script>
</body>
</html>
"""

html = (HTML_TEMPLATE
        .replace("__CONFIG__", json.dumps(_JS_CONFIG))
        .replace("__MERMAID__", json.dumps(MERMAID_SRC)))
print("Assembled HTML: %d chars, %d overlay layers, %d S2 backgrounds." % (
    len(html), len(registry), len(S2_BACKGROUNDS)))


## Render the viewer

The viewer is shown inline below inside a fixed-height iframe (so the embedded Leaflet / Mermaid scripts run in their own document), and is also written to `timber_pathway_viewer.html`, which is downloaded to your machine (in Colab) or reported as a local path otherwise.

> **If the panels are blank inline** (some environments, e.g. VSCode, strip embedded scripts): open the saved `timber_pathway_viewer.html` in a browser - it renders the full viewer.

In [ ]:
import html as _html

OUTPUT_HTML = 'timber_pathway_viewer.html'
with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)
print('Wrote', OUTPUT_HTML, '(' + str(len(html)) + ' chars)')
print('Tile layers embedded:', len(registry), 'overlays +', len(S2_BACKGROUNDS), 'S2 backgrounds')

# Show inline inside a FIXED-HEIGHT iframe via srcdoc, so the embedded Leaflet / Mermaid scripts
# run in their own document and the panes get a real height (a bare display(HTML(...)) gets its
# scripts stripped and its containers collapsed in some notebook sandboxes, e.g. VSCode).
_iframe = (
    '<iframe srcdoc="' + _html.escape(html)
    + '" style="width:100%;height:850px;border:0;"></iframe>'
)
display(HTML(_iframe))

# Deliver the standalone file to the user's machine, matching the other Whisp example notebooks
# (Colab_whisp_geojson_to_csv.ipynb uses `from google.colab import files; files.download(path)`).
# Guarded so it no-ops gracefully outside Colab (just reports the local saved path).
try:
    from google.colab import files  # type: ignore
    files.download(OUTPUT_HTML)
    print('Download started:', OUTPUT_HTML)
except ImportError:
    import os
    print('Not running in Colab; open the saved file from:', os.path.abspath(OUTPUT_HTML))